<a href="https://colab.research.google.com/github/codeunitdev/natlas-devkit/blob/main/notebooks/N_ATLAS_DevKit_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# N-ATLaS DevKit — Colab demo

Run Nigeria's open language model **N-ATLaS** on a free Colab GPU, then use it three ways:
the **Python SDK**, an **OpenAI-compatible API**, and a **web playground**.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Sign in at huggingface.co and accept the terms on the [NCAIR1/N-ATLaS](https://huggingface.co/NCAIR1/N-ATLaS) model page.
3. Create a *read* token at huggingface.co/settings/tokens and add it in Colab's **🔑 Secrets** panel as `HF_TOKEN`.

*N-ATLaS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.*

## 1. Install

In [1]:
!pip install -q "natlas-devkit[all] @ git+https://github.com/codeunitdev/natlas-devkit.git" openai

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.3 MB/s eta 0:00:00


## 2. Hugging Face login

In [2]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

## 3. Load N-ATLaS in 4-bit (about 6 GB of GPU memory; first download takes a few minutes)

In [3]:
from natlas import NAtlas
llm = NAtlas(quantization="4bit")
print("Loaded", llm.model_id)

config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded NCAIR1/N-ATLaS


## 4. Python SDK — one model, four languages

In [4]:
prompts = {
    "Hausa":   "menene ake nufi da gwagwarmaya",
    "Yoruba":  "Báwo ni mo ṣe lè dáàbò bo ara mi lọ́wọ́ ibà?",
    "Igbo":    "Kedu ka m ga-esi chebe onwe m pụọ n'ọrịa ịba?",
    "English": "Explain in simple terms how a farmer can save money with a cooperative.",
}
for lang, q in prompts.items():
    print(f"=== {lang} ===\nQ: {q}\nA: {llm.ask(q, max_new_tokens=250)}\n")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


=== Hausa ===
Q: menene ake nufi da gwagwarmaya
A: Gwagwarmaya tana nufin kokari, juriya, ko kuma yaki na ci gaba don cimma wata manufa, samun nasara, ko shawo kan wahala. Yakan haɗa da fuskantar kalubale masu mahimmanci da fafutuka da suka dace don burin mutum ko al'umma.

=== Yoruba ===
Q: Báwo ni mo ṣe lè dáàbò bo ara mi lọ́wọ́ ibà?
A: Lati daabobo ararẹ kuro ninu iba, rii daju pe o n gba oorun to peye, jijẹ awọn ounjẹ ilera nigbagbogbo, ki o si yago fun mimu barasa tabi awọn ohun mimu ti o mu ki ikun di. Ti o ba n ṣiṣẹ, wọ aṣọ ti o tọ ati bata bata lati ṣetọju iwọn otutu ara. Ni afikun, mimi sinu ati jade lati awọn agbegbe ti o kun le ṣe iranlọwọ lati pa ipele suga ẹjẹ duro.

=== Igbo ===
Q: Kedu ka m ga-esi chebe onwe m pụọ n'ọrịa ịba?
A: Iji chebe onwe gị pụọ n’ọrịa ịba, tụlee ihe ndị a: 

1. Na-aṅụ mmiri buru ibu ma na-edobe ahụ́ ike.
2. Na-asa aka gị mgbe niile, karịsịa mgbe ị gachara ụlọ ịsa ahụ, tupu iri nri, na mgbe ị kpasuru onwe gị.
3. Jiri ncha na mmiri dị nro mee ihe mak

### Translation and forced reply language

In [5]:
print(llm.translate("Wash your hands with soap before eating.", to="Hausa"))
print(llm.ask("What are the early signs of malaria?", language="Igbo", max_new_tokens=200))

Ku wanke hannayenku da sabulu kafin ku ci abinci.
Ihe mbụ nke ịba ọcha bụ ike ọgwụgwụ na-abịa na mberede, mgbu n'isi, ọkụ, na ahụ gị na-enwe mmetụta siri ike ma ọ bụ "na-arịa." Ihe ndị a nwere ike ime ngwa ngwa ka e kparachara nje malaria n'ime ahụ.


### Streaming

In [6]:
for piece in llm.stream_chat([{"role": "user", "content": "Give me three tips for saving money as a student."}], max_new_tokens=250):
    print(piece, end="", flush=True)

1. Create a budget and track your expenses to understand where your money is going.
2. Cook meals in bulk and freeze leftovers to save time and money.
3. Prioritize your spending by focusing on essential needs over wants.

## 5. OpenAI-compatible API

Start the API server in the background (reusing the loaded model), then call it with the **official OpenAI client**.
Any app built for OpenAI switches to N-ATLaS by changing `base_url`.

In [7]:
import threading, time, uvicorn
from natlas.server import create_app

server = uvicorn.Server(uvicorn.Config(create_app(engine=llm), host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

from openai import OpenAI
client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local")
resp = client.chat.completions.create(
    model="n-atlas",
    messages=[{"role": "user", "content": "menene ake nufi da gwagwarmaya"}],
    max_tokens=200,
)
print(resp.choices[0].message.content)
print(resp.usage)

Gwagwarmayar na nufin kokari mai wahala ko kuma yaki da wani abu don cimma burinsa ko shawo kan kalubale. Yakan haɗa da fuskantar matsaloli tare da juriya, azama, da sadaukarwa don samun nasara ko ci gaba.
CompletionUsage(completion_tokens=80, prompt_tokens=90, total_tokens=170, completion_tokens_details=None, prompt_tokens_details=None)


## 6. Web playground (opens a temporary public link)

In [8]:
from natlas.playground import build_demo
build_demo(engine=llm).queue().launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://04b3e5b6f31b2e161b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
